In [12]:
# ANUGA IMPORT DIAGNOSTIC + STRUCTURES MONKEY-PATCH
# The installed anuga lacks structures, but local repo lacks compiled extensions.
# Solution: Use installed anuga + import structures from local repo.

print('--- ANUGA IMPORT DIAGNOSTIC ---')
import sys, importlib, pathlib

# Import the installed anuga package (has compiled extensions)
try:
    import anuga
    print('anuga module:', anuga)
    print('anuga.__file__:', anuga.__file__)
    print("hasattr(anuga, 'structures'):", hasattr(anuga, 'structures'))
except Exception as e:
    print('Importing anuga failed:', e)
    raise RuntimeError('Cannot import anuga from site-packages') from e

# If structures is missing, load it from local repo
if not hasattr(anuga, 'structures'):
    print('\n🔧 Monkey-patching anuga.structures from local repo...')
    local_repo = pathlib.Path(r'e:\4th year\ANUGA-ing\anuga_core-main').resolve()
    
    # Add local repo to sys.path temporarily to import structures
    if str(local_repo) not in sys.path:
        sys.path.insert(0, str(local_repo))
        print(f'   Added local repo to sys.path: {local_repo}')
    
    try:
        # Import structures module from local repo
        import anuga.structures as structures_module
        
        # Attach it to the installed anuga package
        anuga.structures = structures_module
        sys.modules['anuga.structures'] = structures_module
        
        print(f'   ✓ Imported anuga.structures from: {structures_module.__file__}')
        print(f"   ✓ hasattr(anuga, 'structures'): {hasattr(anuga, 'structures')}")
        
        # Also import the key classes that might be needed
        from anuga.structures.riverwall import Riverwall
        anuga.Riverwall = Riverwall
        print(f'   ✓ Added anuga.Riverwall')
        
    except Exception as e:
        print(f'   ✗ Failed to import structures from local repo: {e}')
        import traceback
        traceback.print_exc()
        raise RuntimeError('Cannot import anuga.structures from local repo') from e

# Final check
if not hasattr(anuga, 'structures'):
    raise RuntimeError('anuga.structures still not available after monkey-patch')

print('\n✓ ANUGA diagnostic complete - structures module is accessible via monkey-patch.')


--- ANUGA IMPORT DIAGNOSTIC ---
anuga module: <module 'anuga' from '/home/sentoki/miniconda3/envs/flower/lib/python3.12/site-packages/anuga/__init__.py'>
anuga.__file__: /home/sentoki/miniconda3/envs/flower/lib/python3.12/site-packages/anuga/__init__.py
hasattr(anuga, 'structures'): True

✓ ANUGA diagnostic complete - structures module is accessible via monkey-patch.


# ANUGA Rainfall-Runoff Simulation with GPU Support

This notebook simulates long-duration rainfall events using ANUGA with optional GPU acceleration.

## Features:
- **GPU Support**: Uses CuPy for GPU acceleration (if available)
- **Long Simulations**: Optimized for extended simulation times
- **Checkpointing**: Automatic saving of simulation state for recovery
- **Memory Management**: Efficient handling of large domains
- **Progress Monitoring**: Real-time statistics and visualization

## Quick Start Guide

### For Long Simulations:
1. **Run Cell 1-3** to set up the environment and configuration
2. **Adjust settings in Cell 3** (see below) for your simulation duration
3. **Run Cell 4** to create the domain and enable GPU (if available)
4. **Run Cell 5** to execute the simulation (this is the long-running cell)
5. **Run Cell 6-8** to visualize results

### Configuration Options (in Cell 3):
```python
config.total_simulation_hours = 24.0  # Set to desired duration (e.g., 48, 72, 168 for 1 week)
config.rainfall_duration_hours = 1.0   # How long rain falls
config.rainfall_intensity_mm_hr = 10.0 # Rainfall intensity
config.output_interval_minutes = 30.0  # How often to save results
config.checkpoint_interval_hours = 2.0 # How often to save recovery points
config.maximum_triangle_area = 50000   # Smaller = more accurate but slower
```

### Performance Tips:
- **GPU**: Install CuPy for 5-10x speedup (see GPU Setup cell)
- **Mesh Size**: Increase `maximum_triangle_area` for faster simulation
- **Output Interval**: Larger intervals = less disk I/O
- **Checkpoints**: Enable for long runs so you can resume if interrupted

## ⚠️ IMPORTANT: GPU Configuration Update

**Changes made to enable proper GPU acceleration:**

1. **Added Flow Algorithm Setup** - ANUGA's GPU mode requires `set_flow_algorithm('DE0')` and `set_low_froude(0)` to be compatible with CUDA kernels

2. **Added GPU Verification Test** - New cell after domain setup that:
   - Tests actual GPU kernel execution
   - Runs a tiny evolution to trigger kernel compilation
   - Monitors GPU memory allocation
   - Confirms GPU is truly active

3. **Enhanced Progress Monitoring** - Simulation cell now shows:
   - Current GPU memory usage during simulation
   - Clear indicators when GPU mode is active
   - Instructions to use `nvidia-smi` for verification

**Why `nvidia-smi` showed no Python process before:**
- CuPy was installed and could create arrays
- But ANUGA wasn't actually using GPU for computation
- Likely missing the `DE0` flow algorithm requirement
- Or GPU interface wasn't properly initialized

**To verify GPU is working:**
1. Run the domain setup cell (Cell 9)
2. Run the new GPU verification cell (Cell 10)
3. **Open a terminal and run:** `nvidia-smi -l 1`
4. Run the simulation cell (Cell 11)
5. You should see Python process with GPU memory usage in nvidia-smi

If you still see no Python process in `nvidia-smi`, the GPU is NOT being used.

In [13]:
import anuga 
import numpy as np
import matplotlib.pyplot as plt
import os
import sys
import time

# Check for GPU support
GPU_AVAILABLE = True
try:
    import cupy as cp
    GPU_AVAILABLE = True
    print("✓ CuPy found - GPU acceleration ENABLED")
    print(f"  GPU: {cp.cuda.Device()}")
    print(f"  CuPy version: {cp.__version__}")
except ImportError:
    print("✗ CuPy not found - Running on CPU only")
    print("  To enable GPU: pip install cupy-cuda11x (or cupy-cuda12x for CUDA 12)")
    print("  Note: Requires NVIDIA GPU with CUDA installed")

print(f"ANUGA version: {anuga.__version__ if hasattr(anuga, '__version__') else 'Unknown'}")
print(f"NumPy version: {np.__version__}")

✓ CuPy found - GPU acceleration ENABLED
  GPU: <CUDA Device 0>
  CuPy version: 13.6.0
ANUGA version: 3.2.1rc17
NumPy version: 2.3.5


In [14]:
# CHECK GPU AND SYSTEM CAPABILITIES
import platform

print("="*70)
print("SYSTEM INFORMATION")
print("="*70)
print(f"OS: {platform.system()} {platform.release()}")
print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__}")

# Check for GPU
if GPU_AVAILABLE:
    print("\n" + "="*70)
    print("GPU INFORMATION")
    print("="*70)
    try:
        import cupy as cp
        print(f"CuPy version: {cp.__version__}")
        print(f"GPU Device: {cp.cuda.Device()}")
        
        # Get GPU memory info
        mempool = cp.get_default_memory_pool()
        print(f"GPU Memory - Used: {mempool.used_bytes()/1e9:.2f} GB")
        print(f"GPU Memory - Total: {mempool.total_bytes()/1e9:.2f} GB")
        
        # CUDA version
        print(f"CUDA Version: {cp.cuda.runtime.runtimeGetVersion()}")
        print(f"CUDA Compute Capability: {'.'.join(map(str, cp.cuda.Device().compute_capability))}")
        
        # Number of GPUs
        n_gpus = cp.cuda.runtime.getDeviceCount()
        print(f"Available GPUs: {n_gpus}")
        
        print("\n✓ GPU acceleration is READY")
    except Exception as e:
        print(f"GPU check error: {e}")
else:
    print("\n✗ GPU not available - will use CPU")
    print("  For GPU support, install CuPy matching your CUDA version")

print("="*70)

SYSTEM INFORMATION
OS: Linux 6.14.0-35-generic
Python: 3.12.0
NumPy: 2.3.5

GPU INFORMATION
CuPy version: 13.6.0
GPU Device: <CUDA Device 0>
GPU Memory - Used: 0.01 GB
GPU Memory - Total: 0.01 GB
CUDA Version: 11080
CUDA Compute Capability: 8.6
Available GPUs: 1

✓ GPU acceleration is READY


In [15]:
import os
import sys

# --- GDAL WINDOWS FIX ---
# Anaconda on Windows often hides the GDAL DLLs in the Library\bin folder.
# We must add this to the system path so Python can find '_gdal'.

# 1. Get the absolute path to your current environment's 'Library/bin'
conda_library_bin = os.path.join(sys.prefix, 'Library', 'bin')

# 2. Add it to the system PATH environment variable
if os.path.exists(conda_library_bin):
    os.environ['PATH'] = conda_library_bin + os.pathsep + os.environ['PATH']
    
    # 3. For Python 3.8+, we might also need this explicit call:
    if hasattr(os, 'add_dll_directory'):
        try:
            os.add_dll_directory(conda_library_bin)
        except Exception:
            pass
else:
    print(f"WARNING: Could not find Anaconda Library/bin at: {conda_library_bin}")
# ------------------------



## GPU Setup (Optional but Recommended)

To enable GPU acceleration for faster simulations, install CuPy:

```bash
# For CUDA 11.x
pip install cupy-cuda11x

# For CUDA 12.x
pip install cupy-cuda12x
```

**Requirements:**
- NVIDIA GPU with CUDA support
- CUDA Toolkit installed
- Matching CuPy version for your CUDA version

**Benefits:**
- 5-10x faster computation for large domains
- Enables much longer simulation times
- Better performance with high-resolution meshes

In [16]:
import pandas as pd
from shapely import wkt
import anuga
import os
from anuga import Domain, Geo_reference
from osgeo import gdal, osr
import numpy as np

# --- HELPER FUNCTION ---
def get_tiff_details(tiff_path):
    """Extract zone, proj4, EPSG, and hemisphere from a GeoTIFF."""
    if not os.path.exists(tiff_path): 
        raise FileNotFoundError(f"{tiff_path}")
    ds = gdal.Open(tiff_path)
    if ds is None: 
        raise ValueError("GDAL could not open the file.")
    
    prj = ds.GetProjection()
    srs = osr.SpatialReference(wkt=prj)
    
    if srs.IsProjected():
        zone = srs.GetUTMZone()
        # Fallback for zone parsing
        if zone == 0:
            name = srs.GetAttrValue("PROJCS")
            if name:
                import re
                match = re.search(r"zone\s+(\d+)", name, re.IGNORECASE)
                if match: zone = int(match.group(1))
        
        proj4_str = srs.ExportToProj4()
        epsg = srs.GetAuthorityCode(None)
        
        # Determine hemisphere from EPSG
        hemisphere = 'northern'
        if epsg:
            if epsg.startswith('327'):
                hemisphere = 'southern'
            elif epsg.startswith('326'):
                hemisphere = 'northern'
        
        return zone, proj4_str, epsg, hemisphere
    else:
        raise ValueError("TIFF is not projected.")

def create_tiff_with_epsg(input_tiff, output_tiff, epsg_code):
    """Create a copy of the TIFF with explicit EPSG code."""
    print(f"Creating TIFF with EPSG:{epsg_code}: {output_tiff}")
    try:
        srs = osr.SpatialReference()
        srs.ImportFromEPSG(int(epsg_code))
        
        src_ds = gdal.Open(input_tiff)
        driver = gdal.GetDriverByName('GTiff')
        dst_ds = driver.CreateCopy(output_tiff, src_ds, strict=0)
        dst_ds.SetProjection(srs.ExportToWkt())
        
        dst_ds = None
        src_ds = None
        
        print(f"✓ Created corrected TIFF: {output_tiff}")
        return True
    except Exception as e:
        print(f"✗ Failed to create corrected TIFF: {e}")
        return False

class TopographyLoader:
    def __init__(self, domain):
        self.domain = domain

    def load_geotiff(self, tiff_path: str, quantity_name: str = 'elevation'):
        if not os.path.exists(tiff_path): 
            raise FileNotFoundError(f"{tiff_path}")
        print(f"Loading GeoTIFF: {tiff_path} into '{quantity_name}'...")
        try:
            self.domain.get_quantity(quantity_name).set_values_from_tif_file(tiff_path)
            print("✓ Topography loaded successfully.")
        except Exception as e:
            print(f"✗ Failed to load GeoTIFF. Reason: {e}")
            raise

class AnugaGeometryLoader:
    def parse_qgis_bounding_box(self, csv_path: str, buffer_distance: float = -5000.0) -> list:
        """Parse QGIS bounding box from CSV with optional buffering."""
        df = pd.read_csv(csv_path)
        wkt_string = df['WKT'].iloc[0]
        polygon_geom = wkt.loads(wkt_string)
        
        # Buffer to avoid NoData edges (negative = inward buffer)
        if buffer_distance != 0:
            polygon_geom = polygon_geom.buffer(buffer_distance) 

        coords_tuples = list(polygon_geom.exterior.coords)
        anuga_polygon = [list(pt) for pt in coords_tuples]
        
        if anuga_polygon[0] == anuga_polygon[-1]:
            anuga_polygon.pop()
        return anuga_polygon

class SimulationConfig:
    """Configuration class for simulation parameters."""
    def __init__(self):
        # Files
        self.csv_file = "bounder.csv"
        self.geotiff_file = "dem_UTM.tif"
        self.corrected_tiff = "DEM_UTM_EPSG32643.tif"
        self.target_epsg = "32643"  # UTM Zone 43N
        
        # Domain parameters
        self.maximum_triangle_area = 25000  # m² (larger = faster, less accurate)
        self.buffer_distance = -5000.0  # meters (negative = inward)
        
        # Physical parameters
        self.friction_coefficient = 0.03  # Manning's n
        self.minimum_storable_height = 0.00001  # meters
        
        # Simulation parameters
        self.rainfall_intensity_mm_hr = 10.0  # mm/hr
        self.rainfall_duration_hours = 1.0  # hours
        self.total_simulation_hours = 24.0  # hours (can be very long)
        self.output_interval_minutes = 30.0  # minutes
        
        # Performance parameters
        self.use_gpu = GPU_AVAILABLE  # Auto-detect
        self.save_checkpoints = True
        self.checkpoint_interval_hours = 2.0  # hours
        
        # Output
        self.output_dir = 'outputs'
        self.simulation_name = 'rainfall_simulation'
    
    def get_rainfall_rate_mps(self):
        """Convert rainfall intensity to m/s."""
        return self.rainfall_intensity_mm_hr / (3600.0 * 1000.0)
    
    def print_summary(self):
        """Print configuration summary."""
        print("\n" + "="*70)
        print("SIMULATION CONFIGURATION")
        print("="*70)
        print(f"Domain mesh resolution: {self.maximum_triangle_area} m² per triangle")
        print(f"Rainfall: {self.rainfall_intensity_mm_hr} mm/hr for {self.rainfall_duration_hours} hours")
        print(f"Total simulation time: {self.total_simulation_hours} hours")
        print(f"Output interval: {self.output_interval_minutes} minutes")
        print(f"GPU acceleration: {'ENABLED' if self.use_gpu else 'DISABLED'}")
        print(f"Checkpointing: {'ENABLED' if self.save_checkpoints else 'DISABLED'}")
        if self.save_checkpoints:
            print(f"  Checkpoint interval: {self.checkpoint_interval_hours} hours")
        print("="*70)

# Initialize configuration
config = SimulationConfig()
config.print_summary()


SIMULATION CONFIGURATION
Domain mesh resolution: 25000 m² per triangle
Rainfall: 10.0 mm/hr for 1.0 hours
Total simulation time: 24.0 hours
Output interval: 30.0 minutes
GPU acceleration: ENABLED
Checkpointing: ENABLED
  Checkpoint interval: 2.0 hours


In [17]:
# ADJUST CONFIGURATION HERE FOR YOUR SIMULATION
# Modify these values before running the simulation

# Example configurations for different scenarios:

# Short test run (15 minutes)
config.total_simulation_hours = 0.25
config.rainfall_duration_hours = 0.083  # 5 minutes
config.output_interval_minutes = 1.0

# Standard run (24 hours) - DEFAULT
# config.total_simulation_hours = 24.0
# config.rainfall_duration_hours = 1.0
# config.output_interval_minutes = 30.0
# config.checkpoint_interval_hours = 2.0

# Long run (1 week)
# config.total_simulation_hours = 168.0  # 7 days
# config.rainfall_duration_hours = 6.0   # 6 hour storm
# config.output_interval_minutes = 60.0  # 1 hour output
# config.checkpoint_interval_hours = 12.0 # 12 hour checkpoints

# Very long run (1 month) - for research/climate studies
# config.total_simulation_hours = 720.0  # 30 days
# config.rainfall_duration_hours = 12.0
# config.output_interval_minutes = 120.0  # 2 hour output
# config.checkpoint_interval_hours = 24.0  # daily checkpoints

# Mesh resolution (affects speed and accuracy)
# config.maximum_triangle_area = 100000  # Coarse/fast
# config.maximum_triangle_area = 50000   # Medium (default)
# config.maximum_triangle_area = 25000   # Fine/slow
config.maximum_triangle_area = 10000    # Very Fine (High GPU Load)

print("Current configuration:")
config.print_summary()

Current configuration:

SIMULATION CONFIGURATION
Domain mesh resolution: 10000 m² per triangle
Rainfall: 10.0 mm/hr for 0.083 hours
Total simulation time: 0.25 hours
Output interval: 1.0 minutes
GPU acceleration: ENABLED
Checkpointing: ENABLED
  Checkpoint interval: 2.0 hours


In [18]:
if __name__ == "__main__":
    # 1. GET TIFF CRS with hemisphere
    print("\n" + "="*70)
    print("STEP 1: READING GEOTIFF PROJECTION")
    print("="*70)
    zone_num, tiff_proj4, epsg_code, hemisphere = get_tiff_details(config.geotiff_file)
    print(f"✓ TIFF Zone: {zone_num}, EPSG: {epsg_code}, Hemisphere: {hemisphere}")
    
    # 2. Create corrected TIFF with explicit EPSG
    print("\n" + "="*70)
    print("STEP 2: PREPARING GEOTIFF")
    print("="*70)
    if not os.path.exists(config.corrected_tiff):
        create_tiff_with_epsg(config.geotiff_file, config.corrected_tiff, config.target_epsg)
    else:
        print(f"✓ Using existing corrected TIFF: {config.corrected_tiff}")
    
    # Verify the corrected TIFF
    _, _, corrected_epsg, corrected_hemisphere = get_tiff_details(config.corrected_tiff)
    print(f"✓ Corrected TIFF - EPSG: {corrected_epsg}, Hemisphere: {corrected_hemisphere}")
    
    # 3. LOAD POLYGON
    print("\n" + "="*70)
    print("STEP 3: LOADING BOUNDARY POLYGON")
    print("="*70)
    geo_loader = AnugaGeometryLoader() 
    bounding_polygon = geo_loader.parse_qgis_bounding_box(config.csv_file, config.buffer_distance)
    print(f"✓ Parsed {len(bounding_polygon)} vertices")
    print(f"  First 3 vertices: {bounding_polygon[:3]}")

    # 4. CREATE DOMAIN
    print("\n" + "="*70)
    print("STEP 4: CREATING COMPUTATIONAL MESH")
    print("="*70)
    num_segments = len(bounding_polygon)
    tags = {'exterior': list(range(num_segments))}

    print("Creating mesh (this may take a moment)...")
    start_time = time.time()
    domain = anuga.create_domain_from_regions(
        bounding_polygon, 
        boundary_tags=tags, 
        maximum_triangle_area=config.maximum_triangle_area
    )
    mesh_time = time.time() - start_time
    
    print(f"✓ Domain created with {domain.get_number_of_triangles():,} triangles in {mesh_time:.2f}s")
    print(f"  Domain area: {domain.get_area()/1e6:.2f} km²")
    print(f"  Initial geo_reference: {domain.geo_reference}")
    
    # IMPORTANT: Set flow algorithm for GPU compatibility
    # DE0 is required for GPU acceleration
    print("\n" + "="*70)
    print("STEP 4b: CONFIGURING FLOW ALGORITHM")
    print("="*70)
    domain.set_flow_algorithm('DE0')
    domain.set_low_froude(0)
    print("✓ Flow algorithm set to 'DE0' (required for GPU)")
    print("✓ Low Froude mode disabled (set to 0)")
    
    # 5. SET ZONE AND HEMISPHERE
    domain.geo_reference.set_zone(zone_num)
    domain.geo_reference.set_hemisphere(corrected_hemisphere)
    print(f"✓ Set zone={zone_num}, hemisphere={corrected_hemisphere}")

    # 6. LOAD TOPOGRAPHY
    print("\n" + "="*70)
    print("STEP 5: LOADING TOPOGRAPHY")
    print("="*70)
    topo_loader = TopographyLoader(domain)
    try:
        topo_loader.load_geotiff(config.corrected_tiff)
        
        elev = domain.get_quantity('elevation')
        print(f"✓ Elevation Range: {elev.get_minimum_value():.2f} to {elev.get_maximum_value():.2f} m")
        
    except Exception as e:
        print(f"✗ Execution halted: {e}")
        raise

    # 7. SETUP INITIAL CONDITIONS
    print("\n" + "="*70)
    print("STEP 6: SETTING INITIAL CONDITIONS")
    print("="*70)
    domain.set_quantity('friction', config.friction_coefficient)
    domain.set_quantity('stage', expression='elevation')  # Dry initial condition
    print(f"✓ Friction coefficient: {config.friction_coefficient}")
    print(f"✓ Initial condition: Dry bed (stage = elevation)")
    
    # 8. SETUP BOUNDARY CONDITIONS
    print("\n" + "="*70)
    print("STEP 7: SETTING BOUNDARY CONDITIONS")
    print("="*70)
    Bo = anuga.Dirichlet_boundary([-5.0, 0.0, 0.0])  # Outflow: stage below bed
    domain.set_boundary({'exterior': Bo})
    print("✓ Boundary condition: Dirichlet outflow on all exterior boundaries")
    
    # 9. CONFIGURE DOMAIN PARAMETERS
    domain.set_minimum_storable_height(config.minimum_storable_height)
    domain.set_name(config.simulation_name)
    domain.set_datadir(config.output_dir)
    
    # Create output directory
    os.makedirs(config.output_dir, exist_ok=True)
    
    # 10. ENABLE GPU IF AVAILABLE
    print("\n" + "="*70)
    print("STEP 8: CONFIGURING GPU ACCELERATION")
    print("="*70)
    if config.use_gpu and GPU_AVAILABLE:
        try:
            # Test CuPy with actual computation to ensure GPU is accessible
            import cupy as cp
            print("Testing GPU with CuPy computation...")
            test_array = cp.random.random((1000, 1000))
            test_result = cp.sum(test_array)
            print(f"✓ GPU computation test passed (result: {float(test_result):.4f})")
            
            # Set multiprocessor mode to 2 for GPU
            # This will call set_gpu_interface() which:
            # 1. Creates GPU_interface object
            # 2. Allocates GPU arrays
            # 3. Compiles GPU kernels
            domain.set_multiprocessor_mode(2)
            
            # Verify GPU interface was created
            if domain.gpu_interface is not None:
                print("✓ GPU acceleration ENABLED (multiprocessor_mode=2)")
                print(f"  GPU interface created: {type(domain.gpu_interface)}")
                print(f"  Domain multiprocessor mode: {domain.get_multiprocessor_mode()}")
                print("\n  GPU will be used for:")
                print("    - compute_fluxes_ext_central")
                print("    - extrapolate_second_order_edge_sw")
                print("    - Other shallow water computations")
                print("\n  ⚠ IMPORTANT: During simulation, check 'nvidia-smi' to verify GPU usage!")
            else:
                raise Exception("GPU interface was not created!")
            
        except Exception as e:
            import traceback
            print(f"✗ GPU setup failed: {e}")
            traceback.print_exc()
            print("  Falling back to CPU mode")
            
            # CPU Fallback with OpenMP
            import multiprocessing
            try:
                num_threads = multiprocessing.cpu_count()
                domain.set_omp_num_threads(num_threads)
                print(f"✓ CPU mode: Using {num_threads} OpenMP threads")
            except Exception as omp_e:
                print(f"  Warning: Could not set OpenMP threads: {omp_e}")
                domain.set_multiprocessor_mode(1)
            
            config.use_gpu = False
    else:
        # Use OpenMP for CPU parallelization
        import multiprocessing
        try:
            num_threads = multiprocessing.cpu_count()
            domain.set_omp_num_threads(num_threads)
            print(f"✓ CPU mode: Using {num_threads} OpenMP threads")
        except:
            domain.set_multiprocessor_mode(1)
            print("✓ CPU mode with OpenMP (multiprocessor_mode=1)")
    
    print("\n" + "="*70)
    print("DOMAIN SETUP COMPLETE")
    print("="*70)
    print(f"Triangles: {domain.get_number_of_triangles():,}")
    print(f"Area: {domain.get_area()/1e6:.2f} km²")
    print(f"Elevation: {elev.get_minimum_value():.2f} to {elev.get_maximum_value():.2f} m")
    print(f"Computation mode: {'GPU (CUDA/CuPy)' if config.use_gpu and domain.get_multiprocessor_mode() == 2 else 'CPU (OpenMP)'}")
    if config.use_gpu and domain.get_multiprocessor_mode() == 2:
        print(f"\n💡 TIP: Open a terminal and run 'nvidia-smi' during the next cell")
        print(f"         to verify GPU is being used during simulation!")
    print("="*70)


STEP 1: READING GEOTIFF PROJECTION
✓ TIFF Zone: 43, EPSG: 32643, Hemisphere: northern

STEP 2: PREPARING GEOTIFF
✓ Using existing corrected TIFF: DEM_UTM_EPSG32643.tif
✓ Corrected TIFF - EPSG: 32643, Hemisphere: northern

STEP 3: LOADING BOUNDARY POLYGON
✓ Parsed 4 vertices
  First 3 vertices: [[504986.60961398506, 3323822.6034833225], [504986.733658998, 3424639.745042332], [590499.2448509432, 3425024.0048475643]]

STEP 4: CREATING COMPUTATIONAL MESH
Creating mesh (this may take a moment)...
Setting omp_num_threads to 1
✓ Domain created with 1,334,205 triangles in 17.47s
  Domain area: 8666.64 km²
  Initial geo_reference: (zone=-1, easting=504986.609614, northing=3323822.603483, hemisphere=undefined)

STEP 4b: CONFIGURING FLOW ALGORITHM
✓ Flow algorithm set to 'DE0' (required for GPU)
✓ Low Froude mode disabled (set to 0)
✓ Set zone=43, hemisphere=northern

STEP 5: LOADING TOPOGRAPHY
Loading GeoTIFF: DEM_UTM_EPSG32643.tif into 'elevation'...
✓ Topography loaded successfully.
✓ Elevati

In [19]:
# VERIFY GPU SETUP AND TEST KERNEL EXECUTION
if config.use_gpu and GPU_AVAILABLE and domain.get_multiprocessor_mode() == 2:
    print("="*70)
    print("GPU VERIFICATION TEST")
    print("="*70)
    
    import cupy as cp
    
    # Check GPU memory before ANUGA operations
    print("\n1. GPU Memory Status:")
    mempool = cp.get_default_memory_pool()
    print(f"   Used: {mempool.used_bytes()/1e9:.3f} GB")
    print(f"   Total: {mempool.total_bytes()/1e9:.3f} GB")
    
    # Check if GPU interface is properly initialized
    print("\n2. GPU Interface Status:")
    if domain.gpu_interface is not None:
        print(f"   ✓ GPU interface exists: {type(domain.gpu_interface).__name__}")
        print(f"   ✓ Multiprocessor mode: {domain.get_multiprocessor_mode()}")
        
        # Check if arrays are allocated
        try:
            if hasattr(domain.gpu_interface, 'gpu_stage'):
                print(f"   ✓ GPU arrays allocated")
            else:
                print(f"   ⚠ GPU arrays may not be allocated yet")
        except:
            print(f"   ⚠ Could not check GPU array status")
    else:
        print(f"   ✗ GPU interface is None!")
    
    # Perform a small test evolution to trigger GPU kernel compilation
    print("\n3. Testing GPU Kernel Execution:")
    print("   Running tiny test evolution to activate GPU kernels...")
    
    try:
        # Save current time
        current_time = domain.get_time()
        
        # Do one tiny timestep
        for t in domain.evolve(yieldstep=0.01, finaltime=0.01):
            break
        
        print(f"   ✓ Test evolution completed (t={t:.4f}s)")
        print(f"   ✓ GPU kernels should now be compiled and loaded")
        print(f"\n   💡 NOW is the time to check 'nvidia-smi' in a terminal!")
        print(f"      You should see a Python process using GPU memory")
        
        # Check GPU memory after evolution
        mempool_after = cp.get_default_memory_pool()
        memory_increase = (mempool_after.used_bytes() - mempool.used_bytes()) / 1e6
        print(f"\n   GPU Memory increase: {memory_increase:.2f} MB")
        if memory_increase > 0:
            print(f"   ✓ GPU memory allocated - kernels are active!")
        else:
            print(f"   ⚠ No GPU memory increase - GPU may not be active")
            
    except Exception as e:
        print(f"   ✗ Test evolution failed: {e}")
        print(f"      GPU may not be properly configured")
    
    print("\n" + "="*70)
    print("GPU verification complete. Proceed to full simulation.")
    print("="*70)
    
else:
    print("GPU not enabled - skipping GPU verification test")
    print(f"Config use_gpu: {config.use_gpu}")
    print(f"GPU available: {GPU_AVAILABLE}")
    if hasattr(domain, 'get_multiprocessor_mode'):
        print(f"Multiprocessor mode: {domain.get_multiprocessor_mode()}")

GPU VERIFICATION TEST

1. GPU Memory Status:
   Used: 0.958 GB
   Total: 0.966 GB

2. GPU Interface Status:
   ✓ GPU interface exists: GPU_interface
   ✓ Multiprocessor mode: 2
   ⚠ GPU arrays may not be allocated yet

3. Testing GPU Kernel Execution:
   Running tiny test evolution to activate GPU kernels...
   ✓ Test evolution completed (t=0.0000s)
   ✓ GPU kernels should now be compiled and loaded

   💡 NOW is the time to check 'nvidia-smi' in a terminal!
      You should see a Python process using GPU memory

   GPU Memory increase: 0.00 MB
   ⚠ No GPU memory increase - GPU may not be active

GPU verification complete. Proceed to full simulation.


In [20]:
# DIAGNOSTIC: Investigate ModuleNotFoundError for sw_domain_cuda
# This cell will inspect sys.path, the installed 'anuga' package location,
# and attempt to import 'anuga.shallow_water.sw_domain_cuda' by adding the
# local repository path or loading the module directly from file.

import sys, os, importlib, importlib.util, traceback
from pathlib import Path

print('\n' + '='*70)
print('IMPORT DIAGNOSTIC FOR anuga.shallow_water.sw_domain_cuda')
print('='*70)

# 1) Show current sys.path (trimmed)
print('\n1) sys.path (first 8 entries):')
for p in sys.path[:8]:
    print('  ', p)

# 2) Show which 'anuga' is currently importable and its file
try:
    import anuga
    anuga_file = getattr(anuga, '__file__', None)
    anuga_version = getattr(anuga, '__version__', None)
    print('\n2) Installed anuga:')
    print('   anuga.__file__ =', anuga_file)
    print('   anuga.__version__ =', anuga_version)
except Exception as e:
    print('\n2) Could not import top-level anuga package:')
    print('   ', e)
    anuga = None

# 3) Try direct import of sw_domain_cuda
module_name = 'anuga.shallow_water.sw_domain_cuda'
try:
    print(f"\n3) Trying importlib.import_module('{module_name}')...")
    mod = importlib.import_module(module_name)
    print('   ✓ Imported via standard import')
    found_via = 'standard'
except Exception as exc:
    print('   ✗ Standard import failed:', type(exc).__name__, exc)
    found_via = None

# 4) If import failed, attempt to add local repo 'anuga_core-main' to sys.path
if found_via is None:
    # Try to locate local repo relative to current working directory
    cwd = Path.cwd()
    candidate = cwd / 'anuga_core-main'
    alt_candidate = cwd / 'anuga_core_main'
    print('\n4) Attempting to add local repo to sys.path...')
    if candidate.exists():
        repo_path = str(candidate)
    elif alt_candidate.exists():
        repo_path = str(alt_candidate)
    else:
        # try parent directories (in case notebook running from subdir)
        repo_path = None
        for up in [cwd, *cwd.parents]:
            p = up / 'anuga_core-main'
            if p.exists():
                repo_path = str(p)
                break
    
    if repo_path is None:
        print('   ✗ Could not find local folder "anuga_core-main" in cwd or parents')
        # Also check a known absolute path based on workspace structure
        possible = r'e:\4th year\ANUGA-ing\anuga_core-main'
        if os.path.exists(possible):
            repo_path = possible
            print('   Found at absolute path:', possible)
    else:
        print('   Found local repo at:', repo_path)

    if repo_path:
        if repo_path not in sys.path:
            sys.path.insert(0, repo_path)
            print('   Inserted repo_path into sys.path')
        else:
            print('   repo_path already in sys.path')

        # Retry import
        try:
            mod = importlib.import_module(module_name)
            print('   ✓ Imported sw_domain_cuda after adding local repo to sys.path')
            found_via = 'repo_sys_path'
        except Exception as exc2:
            print('   ✗ Import still failed after adding repo to sys.path:', type(exc2).__name__, exc2)
            # Try loading by file directly
            try:
                file_path = Path(repo_path) / 'anuga' / 'shallow_water' / 'sw_domain_cuda.py'
                print('   Attempting to load module from file:', file_path)
                if file_path.exists():
                    spec = importlib.util.spec_from_file_location('sw_domain_cuda_local', str(file_path))
                    sw_mod = importlib.util.module_from_spec(spec)
                    spec.loader.exec_module(sw_mod)
                    print('   ✓ Loaded module from file via importlib.util')
                    mod = sw_mod
                    found_via = 'file_loader'
                else:
                    print('   ✗ sw_domain_cuda.py not found at expected file path')
            except Exception as e3:
                print('   ✗ Failed to load sw_domain_cuda from file:', type(e3).__name__, e3)
                traceback.print_exc()

# 5) If module loaded, attempt to get GPU_interface
if found_via:
    print(f"\n5) Module loaded via: {found_via}")
    try:
        GPU_interface = getattr(mod, 'GPU_interface', None)
        if GPU_interface is None:
            print('   ✗ GPU_interface class not found in module')
        else:
            print('   ✓ GPU_interface found in module, type:', GPU_interface)

            # Try to instantiate and allocate arrays (safe, with catches)
            try:
                print('   Trying to instantiate GPU_interface(domain) and allocate arrays...')
                gi = GPU_interface(domain)
                try:
                    gi.allocate_gpu_arrays()
                    print('   ✓ allocate_gpu_arrays() completed')
                except Exception as ealloc:
                    print('   ✗ allocate_gpu_arrays() raised:', type(ealloc).__name__, ealloc)
                    traceback.print_exc()
                try:
                    gi.compile_gpu_kernels()
                    print('   ✓ compile_gpu_kernels() completed (may print warnings)')
                except Exception as ecomp:
                    print('   ✗ compile_gpu_kernels() raised:', type(ecomp).__name__, ecomp)
                    traceback.print_exc()
            except Exception as einst:
                print('   ✗ Instantiating GPU_interface failed:', type(einst).__name__, einst)
                traceback.print_exc()
    except Exception as e:
        print('   Error while accessing GPU_interface:', type(e).__name__, e)
        traceback.print_exc()

else:
    print('\n5) sw_domain_cuda module could not be loaded by any method')
    print('   Next steps:')
    print('     - Ensure your Python environment can import the local source by adding the path containing the top-level "anuga" package to sys.path')
    print('     - Or install the development package into your environment: pip install -e path/to/anuga_core-main')

print('\n' + '='*70)
print('IMPORT DIAGNOSTIC COMPLETE')
print('='*70)



IMPORT DIAGNOSTIC FOR anuga.shallow_water.sw_domain_cuda

1) sys.path (first 8 entries):
   /home/sentoki/miniconda3/envs/flower/lib/python312.zip
   /home/sentoki/miniconda3/envs/flower/lib/python3.12
   /home/sentoki/miniconda3/envs/flower/lib/python3.12/lib-dynload
   
   /home/sentoki/miniconda3/envs/flower/lib/python3.12/site-packages

2) Installed anuga:
   anuga.__file__ = /home/sentoki/miniconda3/envs/flower/lib/python3.12/site-packages/anuga/__init__.py
   anuga.__version__ = 3.2.1rc17

3) Trying importlib.import_module('anuga.shallow_water.sw_domain_cuda')...
   ✓ Imported via standard import

5) Module loaded via: standard
   ✓ GPU_interface found in module, type: <class 'anuga.shallow_water.sw_domain_cuda.GPU_interface'>
   Trying to instantiate GPU_interface(domain) and allocate arrays...
   ✓ allocate_gpu_arrays() completed
   ✓ compile_gpu_kernels() completed (may print warnings)

IMPORT DIAGNOSTIC COMPLETE


## 🔧 Fix GPU Kernel Compilation Path

The diagnostic revealed that `compile_gpu_kernels()` uses a hardcoded Linux path for `cuda_anuga.cu`. This cell patches the method to use the correct local Windows path.

In [21]:
# FIX KERNEL COMPILATION - Patch compile_gpu_kernels to use correct path

print('='*70)
print('🔧 PATCHING GPU KERNEL COMPILATION')
print('='*70)

from pathlib import Path
import sys
import re

# Locate the cuda_anuga.cu file in local repo
cuda_cu_path = Path(r'e:\4th year\ANUGA-ing\anuga_core-main\anuga\shallow_water\cuda_anuga.cu')

if not cuda_cu_path.exists():
    print(f'✗ cuda_anuga.cu not found at: {cuda_cu_path}')
    print('  Searching in workspace...')
    # Try to find it
    workspace = Path(r'e:\4th year\ANUGA-ing')
    found = list(workspace.rglob('cuda_anuga.cu'))
    if found:
        cuda_cu_path = found[0]
        print(f'  ✓ Found at: {cuda_cu_path}')
    else:
        print('  ✗ Could not find cuda_anuga.cu anywhere')
        cuda_cu_path = None

if cuda_cu_path and cuda_cu_path.exists():
    print(f'\n✓ CUDA source file located: {cuda_cu_path}')
    print(f'  File size: {cuda_cu_path.stat().st_size / 1024:.1f} KB')
    
    # Now create a patched compile_gpu_kernels function
    def compile_gpu_kernels_patched(self):
        """Patched version with correct file path and bug fixes."""
        import cupy as cp
        
        print(f'  Reading CUDA source from: {cuda_cu_path}')
        with open(cuda_cu_path, 'r') as f:
            code = f.read()
        
        # Define int64_t using CUDA built-in types (NVRTC doesn't support stdint.h)
        print(f'  Prepending type definitions for int64_t...')
        type_definitions = '''
// Define int64_t for NVRTC (doesn't support stdint.h)
typedef long long int64_t;
typedef unsigned long long uint64_t;

'''
        
        # Fix bug in atomicMin_double function - corrupted type declaration
        print(f'  Fixing bugs in CUDA source code...')
        
        # Fix 1: Corrupted unsigned long long declarations (lines 417-418)
        code = code.replace(
            'unsigned int64_t int64_t int64_t* address_as_ull = (unsigned int64_t int64_t int64_t*) address;',
            'unsigned long long* address_as_ull = (unsigned long long*) address;'
        )
        code = code.replace(
            'unsigned int64_t int64_t int64_t old = *address_as_ull, assumed;',
            'unsigned long long old = *address_as_ull, assumed;'
        )
        
        # Fix 2: atomicAdd with int64_t* - CUDA doesn't support this, cast to unsigned long long
        # Replace: atomicAdd(&num_negative_cells, 1); with proper cast
        code = code.replace(
            'atomicAdd(&num_negative_cells, 1);',
            'atomicAdd((unsigned long long*)&num_negative_cells, 1ULL);'
        )
        
        # Prepend type definitions
        code = type_definitions + code
        
        print(f'  Compiling CUDA kernels with CuPy...')
        self.mod = cp.RawModule(code=code, options=("--std=c++17",),
                                name_expressions=("_cuda_compute_fluxes_loop",
                                                  "_cuda_extrapolate_second_order_edge_sw_loop1",
                                                  "_cuda_extrapolate_second_order_edge_sw_loop2",
                                                  "_cuda_extrapolate_second_order_edge_sw_loop3",
                                                  "_cuda_extrapolate_second_order_edge_sw_loop4",
                                                  "_cuda_update_sw",
                                                  "_cuda_fix_negative_cells_sw",
                                                  "_cuda_protect_against_infinitesimal_and_negative_heights",
                                                  "cft_manning_friction_flat",
                                                  "cft_manning_friction_sloped"
                                                  ))
        
        print(f'  Extracting kernel functions...')
        self.flux_kernel = self.mod.get_function("_cuda_compute_fluxes_loop")
        self.extrapolate_kernel1 = self.mod.get_function("_cuda_extrapolate_second_order_edge_sw_loop1")
        self.extrapolate_kernel2 = self.mod.get_function("_cuda_extrapolate_second_order_edge_sw_loop2")
        self.extrapolate_kernel3 = self.mod.get_function("_cuda_extrapolate_second_order_edge_sw_loop3")
        self.extrapolate_kernel4 = self.mod.get_function("_cuda_extrapolate_second_order_edge_sw_loop4")
        self.update_kernel = self.mod.get_function("_cuda_update_sw")
        self.fix_negative_cells_kernel = self.mod.get_function("_cuda_fix_negative_cells_sw")
        self.protect_kernel = self.mod.get_function("_cuda_protect_against_infinitesimal_and_negative_heights")
        self.manning_flat_kernel = self.mod.get_function("cft_manning_friction_flat")
        self.manning_sloped_kernel = self.mod.get_function("cft_manning_friction_sloped")
        
        print(f'  ✓ All kernels extracted successfully')
    
    # Apply the patch
    try:
        # Load the module again (or get from previous diagnostic)
        if 'mod' in locals() and hasattr(mod, 'GPU_interface'):
            GPU_interface = mod.GPU_interface
        else:
            # Reload from file
            import importlib.util
            spec = importlib.util.spec_from_file_location(
                'sw_domain_cuda_local',
                r'e:\4th year\ANUGA-ing\anuga_core-main\anuga\shallow_water\sw_domain_cuda.py'
            )
            mod = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(mod)
            GPU_interface = mod.GPU_interface
        
        print('\n🔧 Creating GPU interface with patched compile method...')
        
        # Create GPU interface
        gi = GPU_interface(domain)
        
        # Patch the method
        import types
        gi.compile_gpu_kernels = types.MethodType(compile_gpu_kernels_patched, gi)
        
        # Allocate arrays
        print('  Allocating GPU arrays...')
        gi.allocate_gpu_arrays()
        print('  ✓ GPU arrays allocated')
        
        # Compile kernels with patched method
        print('  Compiling GPU kernels...')
        gi.compile_gpu_kernels()
        print('  ✓ GPU kernels compiled!')
        
        # Fix calling convention bug: shallow_water_domain.py calls update_conserved_quantities
        # as a function with (self, timestep), but it's actually a method
        # Create a wrapper that fixes the argument order
        print('  Patching update_conserved_quantities_kernel calling convention...')
        original_update = gi.update_conserved_quantities_kernel
        
        def update_conserved_quantities_wrapper(domain_arg, timestep_arg, **kwargs):
            """Wrapper to fix calling convention mismatch."""
            # domain_arg is passed but we already have self (gi) bound to domain
            # So just call the method with timestep
            return original_update(timestep_arg, **kwargs)
        
        # Replace the method with the wrapper
        gi.update_conserved_quantities_kernel = update_conserved_quantities_wrapper
        
        # Attach to domain
        domain.gpu_interface = gi
        domain.multiprocessor_mode = 2
        
        print('\n✓ GPU interface successfully created and attached to domain!')
        print('  domain.multiprocessor_mode =', domain.get_multiprocessor_mode())
        print('  domain.gpu_interface =', type(domain.gpu_interface))
        
        # Update config
        config.use_gpu = True
        
        # Check GPU memory
        if GPU_AVAILABLE:
            import cupy as cp
            mempool = cp.get_default_memory_pool()
            print(f'\n📊 GPU Memory Status:')
            print(f'  Used: {mempool.used_bytes()/1e6:.1f} MB')
            print(f'  Total: {mempool.total_bytes()/1e6:.1f} MB')
        
    except Exception as e:
        print(f'\n✗ Failed to create GPU interface: {e}')
        import traceback
        traceback.print_exc()
        print('\nFalling back to CPU mode...')
        config.use_gpu = False
        try:
            import multiprocessing
            num_threads = multiprocessing.cpu_count()
            domain.set_omp_num_threads(num_threads)
            print(f'✓ CPU mode with {num_threads} OpenMP threads')
        except:
            pass

else:
    print('\n✗ Cannot proceed without cuda_anuga.cu file')
    config.use_gpu = False

print('\n' + '='*70)
print('GPU SETUP COMPLETE')
print('='*70)

🔧 PATCHING GPU KERNEL COMPILATION
✗ cuda_anuga.cu not found at: e:\4th year\ANUGA-ing\anuga_core-main\anuga\shallow_water\cuda_anuga.cu
  Searching in workspace...
  ✗ Could not find cuda_anuga.cu anywhere

✗ Cannot proceed without cuda_anuga.cu file

GPU SETUP COMPLETE


## 🔬 Deep GPU Diagnostics

This cell performs comprehensive GPU diagnostics to identify why GPU is not being used despite being enabled.

**What this checks:**
1. ANUGA's internal GPU routing - which functions are being called
2. GPU kernel execution paths - are they actually triggering GPU code?
3. Memory transfer patterns - is data moving to/from GPU?
4. Function call tracing - what's happening during evolve()

**Run this cell AFTER domain setup and BEFORE the simulation.**

In [22]:
# DEEP GPU DIAGNOSTIC CODE
# This will trace ANUGA's internal calls to determine if GPU is actually being used

print("="*70)
print("🔬 DEEP GPU DIAGNOSTICS")
print("="*70)

import inspect
import functools

# Check domain configuration
print("\n1. DOMAIN CONFIGURATION:")
print(f"   Multiprocessor mode: {domain.get_multiprocessor_mode()}")
print(f"   GPU interface exists: {domain.gpu_interface is not None}")
print(f"   Flow algorithm: {domain.flow_algorithm}")
print(f"   Number of elements: {domain.number_of_elements:,}")

if domain.gpu_interface is not None:
    print(f"\n2. GPU INTERFACE DETAILS:")
    gi = domain.gpu_interface
    print(f"   Type: {type(gi)}")
    print(f"   GPU arrays allocated: {getattr(gi, 'gpu_arrays_allocated', 'Unknown')}")
    
    # Check if GPU kernels are compiled
    kernel_names = ['flux_kernel', 'extrapolate_kernel1', 'extrapolate_kernel2', 
                    'extrapolate_kernel3', 'extrapolate_kernel4', 'update_kernel',
                    'fix_negative_cells_kernel', 'protect_kernel']
    
    print(f"\n3. GPU KERNEL STATUS:")
    for kernel_name in kernel_names:
        has_kernel = hasattr(gi, kernel_name)
        print(f"   {kernel_name}: {'✓ exists' if has_kernel else '✗ missing'}")
    
    # Check GPU memory allocation
    if GPU_AVAILABLE:
        import cupy as cp
        try:
            mempool = cp.get_default_memory_pool()
            print(f"\n4. GPU MEMORY STATUS:")
            print(f"   Used: {mempool.used_bytes()/1e6:.1f} MB")
            print(f"   Total allocated: {mempool.total_bytes()/1e6:.1f} MB")
            
            # Check if specific GPU arrays exist
            gpu_arrays = ['gpu_stage_centroid_values', 'gpu_xmom_centroid_values', 
                         'gpu_ymom_centroid_values', 'gpu_bed_centroid_values']
            print(f"\n5. GPU ARRAY STATUS:")
            for arr_name in gpu_arrays:
                if hasattr(gi, arr_name):
                    arr = getattr(gi, arr_name)
                    print(f"   {arr_name}: {arr.shape if hasattr(arr, 'shape') else 'exists'}")
                else:
                    print(f"   {arr_name}: ✗ missing")
        except Exception as e:
            print(f"   Error checking GPU memory: {e}")

# Create a wrapper to trace function calls
call_log = []

def trace_gpu_calls(func_name, original_func):
    """Wrapper to log when GPU functions are called."""
    @functools.wraps(original_func)
    def wrapper(*args, **kwargs):
        call_log.append(f"GPU: {func_name}")
        return original_func(*args, **kwargs)
    return wrapper

# Check which compute_fluxes function is actually being used
print(f"\n6. FUNCTION ROUTING CHECK:")
print(f"   Checking which compute_fluxes is being called...")

# Try to determine the actual compute_fluxes function
try:
    # Get the compute_fluxes method
    compute_fluxes_method = domain.compute_fluxes
    
    # Check if it's using GPU or CPU version
    method_module = inspect.getmodule(compute_fluxes_method)
    method_file = inspect.getfile(compute_fluxes_method)
    
    print(f"   compute_fluxes module: {method_module}")
    print(f"   compute_fluxes file: {method_file}")
    
    # Check the source code of compute_fluxes
    try:
        source_lines = inspect.getsource(compute_fluxes_method).split('\n')[:20]
        print(f"\n   First 20 lines of compute_fluxes:")
        for i, line in enumerate(source_lines, 1):
            if 'multiprocessor_mode' in line or 'gpu' in line.lower():
                print(f"   {i:3d}: {line}")
    except:
        print("   Could not extract source code")
        
except Exception as e:
    print(f"   Error inspecting compute_fluxes: {e}")

# Test if GPU kernels are callable
if domain.gpu_interface is not None:
    print(f"\n7. GPU KERNEL CALLABLE TEST:")
    gi = domain.gpu_interface
    
    try:
        # Try to call compute_fluxes_ext_central_kernel
        if hasattr(gi, 'compute_fluxes_ext_central_kernel'):
            print("   Attempting to call compute_fluxes_ext_central_kernel...")
            
            # Copy data to GPU first
            try:
                gi.cpu_to_gpu_centroid_values()
                gi.cpu_to_gpu_edge_values()
                gi.cpu_to_gpu_boundary_values()
                print("   ✓ Data copied to GPU")
            except Exception as e:
                print(f"   ⚠ Warning copying to GPU: {e}")
            
            # Try calling the kernel (without retrieving results)
            try:
                test_timestep = gi.compute_fluxes_ext_central_kernel(
                    domain.timestep, 
                    transfer_from_cpu=False, 
                    transfer_gpu_results=False
                )
                print(f"   ✓ GPU kernel executed! Returned timestep: {test_timestep}")
                
                # Check if GPU memory increased
                if GPU_AVAILABLE:
                    import cupy as cp
                    mempool = cp.get_default_memory_pool()
                    print(f"   GPU memory after kernel call: {mempool.used_bytes()/1e6:.1f} MB")
                
            except Exception as e:
                print(f"   ✗ GPU kernel call failed: {e}")
                import traceback
                traceback.print_exc()
        else:
            print("   ✗ compute_fluxes_ext_central_kernel method not found")
            
    except Exception as e:
        print(f"   Error testing GPU kernel: {e}")

# Patch domain methods to trace calls during evolution
print(f"\n8. INSTALLING CALL TRACER:")
print("   Patching domain methods to log which functions are called...")

if domain.gpu_interface is not None:
    gi = domain.gpu_interface
    
    # Wrap GPU interface methods
    if hasattr(gi, 'compute_fluxes_ext_central_kernel'):
        original_compute = gi.compute_fluxes_ext_central_kernel
        gi.compute_fluxes_ext_central_kernel = trace_gpu_calls('compute_fluxes_ext_central_kernel', original_compute)
    
    if hasattr(gi, 'extrapolate_second_order_edge_sw_kernel'):
        original_extrap = gi.extrapolate_second_order_edge_sw_kernel
        gi.extrapolate_second_order_edge_sw_kernel = trace_gpu_calls('extrapolate_second_order_edge_sw_kernel', original_extrap)
    
    print("   ✓ Tracer installed - GPU calls will be logged to call_log")

print("\n" + "="*70)
print("Diagnostics complete. Run a few evolution steps to see what's called.")
print("Check the 'call_log' variable after evolving to see if GPU functions were used.")
print("="*70)

🔬 DEEP GPU DIAGNOSTICS

1. DOMAIN CONFIGURATION:
   Multiprocessor mode: 2
   GPU interface exists: True
   Flow algorithm: DE0
   Number of elements: 1,334,205

2. GPU INTERFACE DETAILS:
   Type: <class 'anuga.shallow_water.sw_domain_cuda.GPU_interface'>
   GPU arrays allocated: True

3. GPU KERNEL STATUS:
   flux_kernel: ✓ exists
   extrapolate_kernel1: ✓ exists
   extrapolate_kernel2: ✓ exists
   extrapolate_kernel3: ✓ exists
   extrapolate_kernel4: ✓ exists
   update_kernel: ✓ exists
   fix_negative_cells_kernel: ✓ exists
   protect_kernel: ✓ exists

4. GPU MEMORY STATUS:
   Used: 958.0 MB
   Total allocated: 1915.9 MB

5. GPU ARRAY STATUS:
   gpu_stage_centroid_values: (1334205,)
   gpu_xmom_centroid_values: (1334205,)
   gpu_ymom_centroid_values: (1334205,)
   gpu_bed_centroid_values: (1334205,)

6. FUNCTION ROUTING CHECK:
   Checking which compute_fluxes is being called...
   compute_fluxes module: <module 'anuga.shallow_water.shallow_water_domain' from '/home/sentoki/miniconda3

In [23]:
# DEEP CUDA KERNEL DIAGNOSTICS
# Diagnose why CUDA kernel is failing at launch

print("="*70)
print("🔍 DEEP CUDA KERNEL DIAGNOSTICS")
print("="*70)

print("\n1️⃣ CHECKING GPU INTERFACE STATE...")
print(f"   domain.multiprocessor_mode = {domain.get_multiprocessor_mode()}")
print(f"   domain.gpu_interface exists = {hasattr(domain, 'gpu_interface')}")

if hasattr(domain, 'gpu_interface'):
    gi = domain.gpu_interface
    
    # Check all GPU arrays are allocated
    print("\n2️⃣ CHECKING GPU ARRAY ALLOCATION...")
    gpu_array_attrs = [
        'gpu_stage_centroid_values', 'gpu_xmom_centroid_values', 'gpu_ymom_centroid_values',
        'gpu_stage_explicit_update', 'gpu_xmom_explicit_update', 'gpu_ymom_explicit_update',
        'gpu_stage_semi_implicit_update', 'gpu_xmom_semi_implicit_update', 'gpu_ymom_semi_implicit_update'
    ]
    
    all_allocated = True
    for attr in gpu_array_attrs:
        if hasattr(gi, attr):
            arr = getattr(gi, attr)
            if arr is None:
                print(f"   ✗ {attr} is None")
                all_allocated = False
            else:
                print(f"   ✓ {attr}: shape={arr.shape}, dtype={arr.dtype}")
        else:
            print(f"   ✗ {attr} not found")
            all_allocated = False
    
    if all_allocated:
        print(f"\n   ✓ All critical GPU arrays are allocated")
    else:
        print(f"\n   ✗ Some GPU arrays are missing - this will cause kernel failures")
    
    # Check kernels are compiled
    print("\n3️⃣ CHECKING COMPILED KERNELS...")
    kernel_attrs = [
        'flux_kernel', 'extrapolate_kernel1', 'update_kernel', 
        'fix_negative_cells_kernel', 'protect_kernel'
    ]
    
    all_compiled = True
    for attr in kernel_attrs:
        if hasattr(gi, attr):
            kernel = getattr(gi, attr)
            if kernel is None:
                print(f"   ✗ {attr} is None")
                all_compiled = False
            else:
                print(f"   ✓ {attr} compiled")
        else:
            print(f"   ✗ {attr} not found")
            all_compiled = False
    
    if all_compiled:
        print(f"\n   ✓ All critical kernels are compiled")
    else:
        print(f"\n   ✗ Some kernels are missing")
    
    # Check CPU-side arrays match GPU arrays in size
    print("\n4️⃣ CHECKING CPU-GPU ARRAY SIZE CONSISTENCY...")
    
    cpu_gpu_pairs = [
        ('cpu_stage_centroid_values', 'gpu_stage_centroid_values'),
        ('cpu_xmom_centroid_values', 'gpu_xmom_centroid_values'),
        ('cpu_ymom_centroid_values', 'gpu_ymom_centroid_values'),
    ]
    
    size_mismatch = False
    for cpu_attr, gpu_attr in cpu_gpu_pairs:
        if hasattr(gi, cpu_attr) and hasattr(gi, gpu_attr):
            cpu_arr = getattr(gi, cpu_attr)
            gpu_arr = getattr(gi, gpu_attr)
            if cpu_arr is not None and gpu_arr is not None:
                if cpu_arr.shape != gpu_arr.shape:
                    print(f"   ✗ SIZE MISMATCH: {cpu_attr}.shape={cpu_arr.shape} != {gpu_attr}.shape={gpu_arr.shape}")
                    size_mismatch = True
                else:
                    print(f"   ✓ {cpu_attr} matches {gpu_attr}: shape={cpu_arr.shape}")
        else:
            print(f"   ⚠ Cannot compare {cpu_attr} and {gpu_attr} (missing)")
    
    if size_mismatch:
        print(f"\n   ✗ CRITICAL: Array size mismatch will cause kernel crashes!")
    else:
        print(f"\n   ✓ All array sizes match")
    
    # Test simple CuPy operation to verify GPU is working
    print("\n5️⃣ TESTING BASIC GPU OPERATIONS...")
    try:
        import cupy as cp
        test_arr = cp.array([1.0, 2.0, 3.0, 4.0, 5.0])
        test_result = test_arr * 2.0
        test_cpu = test_result.get()
        print(f"   ✓ Basic CuPy operations work: {test_cpu}")
    except Exception as e:
        print(f"   ✗ Basic CuPy operation failed: {e}")
    
    # Try to manually call update kernel with minimal data
    print("\n6️⃣ TESTING UPDATE KERNEL WITH MINIMAL CALL...")
    try:
        import cupy as cp
        import numpy as np
        
        # Get actual array from GPU interface
        if hasattr(gi, 'gpu_stage_centroid_values') and gi.gpu_stage_centroid_values is not None:
            n_elements = len(gi.gpu_stage_centroid_values)
            print(f"   Number of elements: {n_elements}")
            
            # Check if update kernel exists
            if hasattr(gi, 'update_kernel') and gi.update_kernel is not None:
                print(f"   ✓ Update kernel exists")
                
                # Try to get kernel launch configuration
                import math
                THREADS_PER_BLOCK = 128
                NO_OF_BLOCKS = int(math.ceil(n_elements / THREADS_PER_BLOCK))
                print(f"   Grid config: {NO_OF_BLOCKS} blocks × {THREADS_PER_BLOCK} threads")
                
                # Check if this is reasonable
                if NO_OF_BLOCKS > 65535:
                    print(f"   ✗ WARNING: Block count {NO_OF_BLOCKS} exceeds CUDA limit (65535)")
                    print(f"   This will cause launch failures!")
                else:
                    print(f"   ✓ Grid configuration is valid")
                
                # Try a minimal kernel call (just the update kernel with proper parameters)
                print(f"\n   Attempting minimal update_kernel call...")
                try:
                    # Ensure arrays are transferred
                    gi.cpu_to_gpu_centroid_values()
                    gi.cpu_to_gpu_explicit_update()
                    gi.cpu_to_gpu_semi_implicit_update()
                    
                    print(f"   ✓ Arrays transferred to GPU")
                    
                    # Call update kernel
                    gi.update_kernel((NO_OF_BLOCKS, 1, 1), (THREADS_PER_BLOCK, 1, 1), (
                        np.int64(n_elements),
                        np.float64(0.001),  # Small timestep
                        gi.gpu_stage_centroid_values,
                        gi.gpu_stage_explicit_update,
                        gi.gpu_stage_semi_implicit_update
                    ))
                    
                    # Synchronize to catch errors
                    cp.cuda.Stream.null.synchronize()
                    
                    print(f"   ✓ UPDATE KERNEL EXECUTED SUCCESSFULLY!")
                    print(f"   The kernel itself works - issue may be in data preparation or other kernels")
                    
                except Exception as kernel_err:
                    print(f"   ✗ Kernel execution failed: {kernel_err}")
                    print(f"   This is the actual kernel error!")
                    import traceback
                    traceback.print_exc()
                    
            else:
                print(f"   ✗ Update kernel not found or is None")
        else:
            print(f"   ✗ gpu_stage_centroid_values not allocated")
            
    except Exception as e:
        print(f"   ✗ Test failed: {e}")
        import traceback
        traceback.print_exc()

else:
    print("\n✗ domain.gpu_interface not found - GPU setup failed completely")

print("\n" + "="*70)
print("DIAGNOSTIC COMPLETE")
print("="*70)

🔍 DEEP CUDA KERNEL DIAGNOSTICS

1️⃣ CHECKING GPU INTERFACE STATE...
   domain.multiprocessor_mode = 2
   domain.gpu_interface exists = True

2️⃣ CHECKING GPU ARRAY ALLOCATION...
   ✓ gpu_stage_centroid_values: shape=(1334205,), dtype=float64
   ✓ gpu_xmom_centroid_values: shape=(1334205,), dtype=float64
   ✓ gpu_ymom_centroid_values: shape=(1334205,), dtype=float64
   ✓ gpu_stage_explicit_update: shape=(1334205,), dtype=float64
   ✓ gpu_xmom_explicit_update: shape=(1334205,), dtype=float64
   ✓ gpu_ymom_explicit_update: shape=(1334205,), dtype=float64
   ✓ gpu_stage_semi_implicit_update: shape=(1334205,), dtype=float64
   ✓ gpu_xmom_semi_implicit_update: shape=(1334205,), dtype=float64
   ✓ gpu_ymom_semi_implicit_update: shape=(1334205,), dtype=float64

   ✓ All critical GPU arrays are allocated

3️⃣ CHECKING COMPILED KERNELS...
   ✓ flux_kernel compiled
   ✓ extrapolate_kernel1 compiled
   ✓ update_kernel compiled
   ✓ fix_negative_cells_kernel compiled
   ✓ protect_kernel compiled

 

## 🔄 Reset GPU Context

The GPU is in a corrupted error state from a previous kernel crash. This requires resetting the CUDA context.

In [24]:
# RESET CUDA CONTEXT TO RECOVER FROM ERROR STATE

print("="*70)
print("🔄 RESETTING CUDA CONTEXT")
print("="*70)

print("\n⚠️  WARNING: This will clear all GPU memory and reset the GPU state")
print("   You will need to re-run the GPU setup cell after this.")

try:
    import cupy as cp
    
    # Get error status before reset
    print("\n1️⃣ Checking current GPU error state...")
    try:
        test = cp.array([1.0])
        print("   GPU state: OK (no reset needed)")
    except Exception as e:
        print(f"   GPU state: ERROR - {type(e).__name__}")
        print(f"   Error message: {str(e)[:100]}")
    
    # Reset device
    print("\n2️⃣ Resetting CUDA device...")
    cp.cuda.Device(0).synchronize()
    
    # Clear memory pools
    print("   Clearing memory pools...")
    mempool = cp.get_default_memory_pool()
    pinned_mempool = cp.get_default_pinned_memory_pool()
    
    print(f"   GPU memory before: {mempool.used_bytes()/1e6:.1f} MB used, {mempool.total_bytes()/1e6:.1f} MB total")
    
    mempool.free_all_blocks()
    pinned_mempool.free_all_blocks()
    
    print(f"   GPU memory after: {mempool.used_bytes()/1e6:.1f} MB used, {mempool.total_bytes()/1e6:.1f} MB total")
    
    # Test if reset worked
    print("\n3️⃣ Testing GPU after reset...")
    try:
        test_arr = cp.array([1.0, 2.0, 3.0])
        test_result = test_arr * 2.0
        test_cpu = test_result.get()
        print(f"   ✓ GPU is working! Test: {test_cpu}")
        print("\n✓ CUDA context successfully reset")
        print("\n📌 NEXT STEPS:")
        print("   1. Re-run Cell 14 (GPU setup with kernel compilation)")
        print("   2. Re-run Cell 16 (inspect GPU interface)")
        print("   3. Then try the simulation again")
        
    except Exception as test_err:
        print(f"   ✗ GPU still in error state: {test_err}")
        print("\n⚠️  MANUAL RESET REQUIRED:")
        print("   Option 1: Restart the Jupyter kernel (Kernel → Restart)")
        print("   Option 2: Close all Python/Jupyter processes and restart")
        
except Exception as e:
    print(f"\n✗ Reset failed: {e}")
    import traceback
    traceback.print_exc()
    print("\n⚠️  You may need to restart the Jupyter kernel")

print("\n" + "="*70)

🔄 RESETTING CUDA CONTEXT

⚠️  WARNING: This will clear all GPU memory and reset the GPU state
   You will need to re-run the GPU setup cell after this.

1️⃣ Checking current GPU error state...
   GPU state: OK (no reset needed)

2️⃣ Resetting CUDA device...
   Clearing memory pools...
   GPU memory before: 958.0 MB used, 1915.9 MB total
   GPU memory after: 958.0 MB used, 966.0 MB total

3️⃣ Testing GPU after reset...
   ✓ GPU is working! Test: [2. 4. 6.]

✓ CUDA context successfully reset

📌 NEXT STEPS:
   1. Re-run Cell 14 (GPU setup with kernel compilation)
   2. Re-run Cell 16 (inspect GPU interface)
   3. Then try the simulation again



## 🔍 Check ALL GPU Arrays

The kernel crash might be due to missing array allocations. Let's check EVERY array the kernels need.

In [25]:
# CHECK ALL GPU ARRAYS REQUIRED BY KERNELS

print("="*70)
print("🔍 COMPREHENSIVE GPU ARRAY CHECK")
print("="*70)

if hasattr(domain, 'gpu_interface'):
    gi = domain.gpu_interface
    
    # All arrays that kernels might access
    all_required_arrays = [
        # Centroid values
        'gpu_stage_centroid_values',
        'gpu_xmom_centroid_values',
        'gpu_ymom_centroid_values',
        'gpu_height_centroid_values',
        'gpu_bed_centroid_values',
        
        # Edge values  
        'gpu_stage_edge_values',
        'gpu_xmom_edge_values',
        'gpu_ymom_edge_values',
        'gpu_height_edge_values',
        'gpu_bed_edge_values',
        
        # Explicit/implicit updates
        'gpu_stage_explicit_update',
        'gpu_xmom_explicit_update',
        'gpu_ymom_explicit_update',
        'gpu_stage_semi_implicit_update',
        'gpu_xmom_semi_implicit_update',
        'gpu_ymom_semi_implicit_update',
        
        # Geometry/topology
        'gpu_neighbours',
        'gpu_neighbour_edges',
        'gpu_normals',
        'gpu_edgelengths',
        'gpu_radii',
        'gpu_areas',
        
        # Coordinates
        'gpu_centroid_coordinates',
        'gpu_edge_coordinates',
        'gpu_vertex_coordinates',
        
        # Work arrays
        'gpu_x_centroid_work',
        'gpu_y_centroid_work',
        
        # Boundaries
        'gpu_number_of_boundaries',
        'gpu_surrogate_neighbours',
        
        # Flux/edge types
        'gpu_edge_flux_type',
        'gpu_tri_full_flag',
        
        # River walls (if any)
        'gpu_edge_river_wall_counter',
        'gpu_riverwall_rowIndex',
        'gpu_riverwall_hydraulic_properties',
    ]
    
    print(f"\nChecking {len(all_required_arrays)} potential GPU arrays...\n")
    
    missing_arrays = []
    null_arrays = []
    allocated_arrays = []
    
    for attr in all_required_arrays:
        if hasattr(gi, attr):
            arr = getattr(gi, attr)
            if arr is None:
                null_arrays.append(attr)
                print(f"   ⚠️  {attr}: EXISTS but is None")
            else:
                allocated_arrays.append(attr)
                print(f"   ✓ {attr}: shape={arr.shape}, dtype={arr.dtype}")
        else:
            missing_arrays.append(attr)
            print(f"   ✗ {attr}: MISSING")
    
    print(f"\n" + "="*70)
    print(f"SUMMARY:")
    print(f"   ✓ Allocated: {len(allocated_arrays)}")
    print(f"   ⚠️  Null: {len(null_arrays)}")
    print(f"   ✗ Missing: {len(missing_arrays)}")
    print(f"="*70)
    
    if null_arrays or missing_arrays:
        print(f"\n⚠️  CRITICAL ISSUE: Missing or null GPU arrays detected!")
        print(f"\nNull arrays: {null_arrays}")
        print(f"Missing attributes: {missing_arrays}")
        print(f"\n💡 SOLUTION:")
        print(f"   The GPU interface's allocate_gpu_arrays() may not be")
        print(f"   allocating all required arrays. This WILL cause kernel crashes.")
        print(f"\n   Recommendation: Check sw_domain_cuda.py allocate_gpu_arrays() method")
    else:
        print(f"\n✓ All required GPU arrays are allocated!")
        print(f"\n   If kernels still crash, the issue is likely:")
        print(f"   1. Kernel code bugs (already fixed some)")
        print(f"   2. Array not initialized with data before kernel call")
        print(f"   3. Incorrect array dimensions/strides")

else:
    print("✗ No GPU interface found")

print()

🔍 COMPREHENSIVE GPU ARRAY CHECK

Checking 34 potential GPU arrays...

   ✓ gpu_stage_centroid_values: shape=(1334205,), dtype=float64
   ✓ gpu_xmom_centroid_values: shape=(1334205,), dtype=float64
   ✓ gpu_ymom_centroid_values: shape=(1334205,), dtype=float64
   ✓ gpu_height_centroid_values: shape=(1334205,), dtype=float64
   ✓ gpu_bed_centroid_values: shape=(1334205,), dtype=float64
   ✓ gpu_stage_edge_values: shape=(1334205, 3), dtype=float64
   ✓ gpu_xmom_edge_values: shape=(1334205, 3), dtype=float64
   ✓ gpu_ymom_edge_values: shape=(1334205, 3), dtype=float64
   ✓ gpu_height_edge_values: shape=(1334205, 3), dtype=float64
   ✓ gpu_bed_edge_values: shape=(1334205, 3), dtype=float64
   ✓ gpu_stage_explicit_update: shape=(1334205,), dtype=float64
   ✓ gpu_xmom_explicit_update: shape=(1334205,), dtype=float64
   ✓ gpu_ymom_explicit_update: shape=(1334205,), dtype=float64
   ✓ gpu_stage_semi_implicit_update: shape=(1334205,), dtype=float64
   ✓ gpu_xmom_semi_implicit_update: shape=(1334

## 🔍 Additional GPU Verification Tests

If the diagnostics above show GPU functions are being called but `nvidia-smi` still shows no activity, run this cell for additional checks.

In [26]:
# ADDITIONAL GPU VERIFICATION - Check if CuPy is actually using GPU

print("="*70)
print("🔍 CUPY GPU VERIFICATION")
print("="*70)

if GPU_AVAILABLE:
    import cupy as cp
    import time
    
    print("\n1. Testing CuPy GPU computation (should show in nvidia-smi)...")
    print("   Creating large arrays and doing computation...")
    
    # Create large arrays on GPU
    size = 5000
    print(f"   Creating {size}x{size} matrices on GPU...")
    
    start_time = time.time()
    a = cp.random.random((size, size), dtype=cp.float32)
    b = cp.random.random((size, size), dtype=cp.float32)
    
    print("   Performing matrix multiplication on GPU...")
    # Force GPU computation
    c = cp.matmul(a, b)
    cp.cuda.Stream.null.synchronize()  # Wait for GPU to finish
    
    gpu_time = time.time() - start_time
    
    print(f"   ✓ GPU computation completed in {gpu_time:.3f} seconds")
    print(f"   Result shape: {c.shape}, mean value: {float(cp.mean(c)):.4f}")
    
    # Check memory
    mempool = cp.get_default_memory_pool()
    print(f"\n2. GPU Memory after CuPy test:")
    print(f"   Used: {mempool.used_bytes()/1e6:.1f} MB")
    print(f"   Total: {mempool.total_bytes()/1e6:.1f} MB")
    
    print("\n   💡 CHECK nvidia-smi NOW - you should see Python using GPU!")
    print("      If you don't see activity, there's a CuPy/CUDA installation issue.")
    
    # Compare with CPU
    print("\n3. Comparing with CPU computation...")
    import numpy as np
    
    start_time = time.time()
    a_cpu = np.random.random((size, size)).astype(np.float32)
    b_cpu = np.random.random((size, size)).astype(np.float32)
    c_cpu = np.matmul(a_cpu, b_cpu)
    cpu_time = time.time() - start_time
    
    print(f"   CPU computation completed in {cpu_time:.3f} seconds")
    print(f"   GPU speedup: {cpu_time/gpu_time:.2f}x")
    
    if cpu_time/gpu_time < 1.5:
        print("\n   ⚠ WARNING: GPU is not faster than CPU!")
        print("      This suggests GPU is not actually being used.")
        print("      Possible causes:")
        print("      - CuPy is falling back to CPU")
        print("      - CUDA drivers not properly installed")
        print("      - GPU is not selected (multi-GPU systems)")
    else:
        print("\n   ✓ GPU is significantly faster - GPU is working!")
    
    # Clean up
    del a, b, c, a_cpu, b_cpu, c_cpu
    
    # Force garbage collection
    import gc
    gc.collect()
    mempool.free_all_blocks()
    
    print("\n4. Checking ANUGA GPU arrays...")
    if domain.gpu_interface is not None:
        gi = domain.gpu_interface
        
        # Try to get size of GPU arrays
        total_gpu_array_size = 0
        gpu_array_attrs = [attr for attr in dir(gi) if attr.startswith('gpu_')]
        
        print(f"   Found {len(gpu_array_attrs)} GPU array attributes")
        
        sample_arrays = ['gpu_stage_centroid_values', 'gpu_xmom_centroid_values', 
                        'gpu_stage_edge_values', 'gpu_bed_centroid_values']
        
        for arr_name in sample_arrays:
            if hasattr(gi, arr_name):
                arr = getattr(gi, arr_name)
                if hasattr(arr, 'nbytes'):
                    size_mb = arr.nbytes / 1e6
                    total_gpu_array_size += arr.nbytes
                    print(f"   {arr_name}: {size_mb:.2f} MB")
        
        print(f"\n   Total ANUGA GPU array size: {total_gpu_array_size/1e6:.2f} MB")
        
        if total_gpu_array_size < 1e6:  # Less than 1 MB
            print("   ⚠ ANUGA GPU arrays are very small - data may not be on GPU")
        else:
            print("   ✓ ANUGA has allocated significant GPU memory")
    
else:
    print("CuPy not available - cannot run GPU verification tests")

print("\n" + "="*70)
print("VERIFICATION COMPLETE")
print("="*70)

🔍 CUPY GPU VERIFICATION

1. Testing CuPy GPU computation (should show in nvidia-smi)...
   Creating large arrays and doing computation...
   Creating 5000x5000 matrices on GPU...
   Performing matrix multiplication on GPU...
   ✓ GPU computation completed in 0.415 seconds
   Result shape: (5000, 5000), mean value: 1249.7902

2. GPU Memory after CuPy test:
   Used: 1258.0 MB
   Total: 1266.0 MB

   💡 CHECK nvidia-smi NOW - you should see Python using GPU!
      If you don't see activity, there's a CuPy/CUDA installation issue.

3. Comparing with CPU computation...
   CPU computation completed in 0.876 seconds
   GPU speedup: 2.11x

   ✓ GPU is significantly faster - GPU is working!

4. Checking ANUGA GPU arrays...
   Found 55 GPU array attributes
   gpu_stage_centroid_values: 10.67 MB
   gpu_xmom_centroid_values: 10.67 MB
   gpu_stage_edge_values: 32.02 MB
   gpu_bed_centroid_values: 10.67 MB

   Total ANUGA GPU array size: 64.04 MB
   ✓ ANUGA has allocated significant GPU memory

VERIF

## GPU Monitoring Instructions

Before running the simulation cell below, you can monitor GPU usage in real-time:

### Option 1: Windows Terminal
Open a new PowerShell/CMD terminal and run:
```powershell
nvidia-smi -l 1
```
This will update every 1 second showing GPU usage.

### Option 2: From VS Code Terminal
You can also run this in the VS Code terminal while the simulation runs.

### What to Look For:
When GPU is **actually being used**, you should see:
- A **Python process** in the process list
- **GPU Memory Usage** increasing (typically 100-500 MB or more)
- **GPU Utilization %** fluctuating (10-100%)

### Troubleshooting:
If you see **NO Python process** in `nvidia-smi`:
1. The GPU is **not** being used (despite CuPy being available)
2. ANUGA may be falling back to CPU mode silently
3. Check the verification cell output above for errors

The next cell will run a small test to trigger GPU kernel loading.

In [ ]:
# OPTIMIZED SIMULATION WITH RAINFALL, CHECKPOINTING, AND PROGRESS MONITORING
from anuga.operators.rate_operators import Rate_operator
import time
import pickle

print("\n" + "="*70)
print("RAINFALL SIMULATION WITH OPTIMIZATIONS")
print("="*70)

# Convert rainfall intensity to m/s
rainfall_intensity = config.get_rainfall_rate_mps()
rainfall_duration_sec = config.rainfall_duration_hours * 3600
total_simulation_sec = config.total_simulation_hours * 3600
yieldstep = config.output_interval_minutes * 60

print(f"Rainfall: {config.rainfall_intensity_mm_hr} mm/hr = {rainfall_intensity*1000*3600:.6f} mm/hr")
print(f"Rainfall duration: {config.rainfall_duration_hours} hours ({rainfall_duration_sec:.0f} seconds)")
print(f"Total simulation time: {config.total_simulation_hours} hours ({total_simulation_sec:.0f} seconds)")
print(f"Output interval: {config.output_interval_minutes} minutes ({yieldstep:.0f} seconds)")
print(f"Domain area: {domain.get_area()/1e6:.2f} km²")
print(f"Computation mode: {'GPU (mode=2)' if domain.get_multiprocessor_mode() == 2 else 'CPU (mode=1)'}")

# GPU monitoring setup
if domain.get_multiprocessor_mode() == 2 and GPU_AVAILABLE:
    import cupy as cp
    print("\n🔥 GPU MODE ACTIVE - Monitor with 'nvidia-smi' in terminal!")
    print("   You should see Python process using GPU during evolution")
    
print("="*70)

# Create time-dependent rainfall function
def rainfall_rate(t):
    """Rainfall rate as function of time."""
    if t <= rainfall_duration_sec:
        return rainfall_intensity
    else:
        return 0.0

# Apply rainfall over entire domain
rain_op = Rate_operator(domain, rate=rainfall_rate)

# Storage for time series data
time_data = []
max_depth_data = []
total_volume_data = []
mean_depth_data = []
timestep_data = []

# Checkpoint management
checkpoint_interval_sec = config.checkpoint_interval_hours * 3600
next_checkpoint = checkpoint_interval_sec if config.save_checkpoints else float('inf')
checkpoint_count = 0

def save_checkpoint(domain, t, checkpoint_num):
    """Save simulation checkpoint."""
    checkpoint_file = os.path.join(config.output_dir, f'checkpoint_{checkpoint_num:03d}.pkl')
    checkpoint_data = {
        'time': t,
        'stage': domain.get_quantity('stage').get_values(location='centroids').copy(),
        'xmom': domain.get_quantity('xmomentum').get_values(location='centroids').copy(),
        'ymom': domain.get_quantity('ymomentum').get_values(location='centroids').copy(),
        'timestep': domain.timestep,
    }
    with open(checkpoint_file, 'wb') as f:
        pickle.dump(checkpoint_data, f)
    print(f"  ✓ Checkpoint saved: {checkpoint_file}")

def print_progress(t, max_d, mean_d, vol, dt, elapsed, rate=1.0, gpu_mem=None):
    """Print formatted progress update."""
    hours = int(t // 3600)
    mins = int((t % 3600) // 60)
    status = "RAIN" if t <= rainfall_duration_sec else "DRAIN"
    
    progress_str = (f"  [{status:5s}] t={hours:02d}:{mins:02d} ({t/3600:.2f}h) | "
                   f"Max:{max_d:.3f}m | Mean:{mean_d:.3f}m | "
                   f"Vol:{vol/1e6:.2f}M m³ | dt:{dt:.2f}s | "
                   f"Speed:{rate:.1f}x realtime | Elapsed:{elapsed/60:.1f}min")
    
    if gpu_mem is not None:
        progress_str += f" | GPU:{gpu_mem:.2f}GB"
    
    print(progress_str)

# Get elevation for calculations
elev_centroids = domain.get_quantity('elevation').get_values(location='centroids')

# Check if we're in GPU mode for monitoring
using_gpu = domain.get_multiprocessor_mode() == 2 and GPU_AVAILABLE

print("\nStarting simulation...")
if using_gpu:
    print("⚡ GPU ACCELERATION ACTIVE - ANUGA will now use CUDA kernels")
    print("   Run 'nvidia-smi' in terminal NOW to see GPU usage!")
print("="*70)

start_time = time.time()
last_print_time = start_time
iteration_count = 0

# Evolve through time
for t in domain.evolve(yieldstep=yieldstep, finaltime=total_simulation_sec):
    iteration_count += 1
    
    # Get current state
    stage_centroids = domain.get_quantity('stage').get_values(location='centroids')
    
    # Calculate water depth
    depth_centroids = np.maximum(stage_centroids - elev_centroids, 0.0)
    
    max_depth = depth_centroids.max()
    wet_mask = depth_centroids > 0.001
    mean_depth = depth_centroids[wet_mask].mean() if wet_mask.any() else 0.0
    total_volume = depth_centroids.sum() * domain.get_area() / len(depth_centroids)
    
    # Store data
    time_data.append(t)
    max_depth_data.append(max_depth)
    mean_depth_data.append(mean_depth)
    total_volume_data.append(total_volume)
    timestep_data.append(domain.timestep)
    
    # GPU memory monitoring
    gpu_mem_used = None
    if using_gpu and iteration_count % 5 == 0:  # Check every 5 iterations
        try:
            mempool = cp.get_default_memory_pool()
            gpu_mem_used = mempool.used_bytes() / 1e9
        except:
            pass
    
    # Print progress (every 10 minutes of simulation time or every 30 seconds real time)
    current_time = time.time()
    elapsed = current_time - start_time
    if int(t) % 600 == 0 or (current_time - last_print_time) > 30:
        # Calculate simulation speed (sim time / real time)
        speed_ratio = t / elapsed if elapsed > 0 else 0
        print_progress(t, max_depth, mean_depth, total_volume, domain.timestep, 
                      elapsed, speed_ratio, gpu_mem_used)
        last_print_time = current_time
    
    # Save checkpoint if needed
    if config.save_checkpoints and t >= next_checkpoint:
        print(f"\n  Saving checkpoint at t={t/3600:.2f}h...")
        save_checkpoint(domain, t, checkpoint_count)
        checkpoint_count += 1
        next_checkpoint += checkpoint_interval_sec

elapsed_total = time.time() - start_time
speed_ratio_final = total_simulation_sec / elapsed_total

print("\n" + "="*70)
print("SIMULATION COMPLETE")
print("="*70)
print(f"Real time elapsed: {elapsed_total/60:.2f} minutes ({elapsed_total/3600:.2f} hours)")
print(f"Simulated time: {total_simulation_sec/3600:.2f} hours")
print(f"Speed: {speed_ratio_final:.2f}x realtime")
print(f"Computation mode: {'GPU (CUDA)' if using_gpu else 'CPU (OpenMP)'}")
print(f"Final max depth: {max_depth_data[-1]:.4f} m ({max_depth_data[-1]*1000:.2f} mm)")
print(f"Final mean depth: {mean_depth_data[-1]:.4f} m ({mean_depth_data[-1]*1000:.2f} mm)")
print(f"Final volume: {total_volume_data[-1]/1e6:.2f} million m³")
if config.save_checkpoints:
    print(f"Checkpoints saved: {checkpoint_count}")

if using_gpu:
    try:
        final_gpu_mem = cp.get_default_memory_pool().used_bytes() / 1e9
        print(f"GPU memory used: {final_gpu_mem:.3f} GB")
    except:
        pass

print("="*70)

# Save time series data
timeseries_file = os.path.join(config.output_dir, 'timeseries_data.npz')
np.savez(timeseries_file,
         time=np.array(time_data),
         max_depth=np.array(max_depth_data),
         mean_depth=np.array(mean_depth_data),
         total_volume=np.array(total_volume_data),
         timestep=np.array(timestep_data))
print(f"✓ Time series data saved: {timeseries_file}")


RAINFALL SIMULATION WITH OPTIMIZATIONS
Rainfall: 10.0 mm/hr = 10.000000 mm/hr
Rainfall duration: 0.083 hours (299 seconds)
Total simulation time: 0.25 hours (900 seconds)
Output interval: 1.0 minutes (60 seconds)
Domain area: 8666.64 km²
Computation mode: GPU (mode=2)

🔥 GPU MODE ACTIVE - Monitor with 'nvidia-smi' in terminal!
   You should see Python process using GPU during evolution

Starting simulation...
⚡ GPU ACCELERATION ACTIVE - ANUGA will now use CUDA kernels
   Run 'nvidia-smi' in terminal NOW to see GPU usage!


TypeError: float() argument must be a string or a real number, not 'Domain'

: 

In [ ]:
# PLOT TIME SERIES RESULTS
import matplotlib.pyplot as plt

print("\nGenerating time series plots...")

# Convert time to hours
time_hours = np.array(time_data) / 3600.0

# Create comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Maximum depth over time
ax1 = axes[0, 0]
ax1.plot(time_hours, max_depth_data, 'b-', linewidth=2)
ax1.axvline(x=config.rainfall_duration_hours, color='r', linestyle='--', 
            linewidth=1.5, label='Rain stops', alpha=0.7)
ax1.set_xlabel('Time (hours)', fontsize=12)
ax1.set_ylabel('Maximum Depth (m)', fontsize=12)
ax1.set_title('Maximum Water Depth Over Time', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=10)

# Plot 2: Mean depth over time
ax2 = axes[0, 1]
ax2.plot(time_hours, mean_depth_data, 'g-', linewidth=2)
ax2.axvline(x=config.rainfall_duration_hours, color='r', linestyle='--', 
            linewidth=1.5, label='Rain stops', alpha=0.7)
ax2.set_xlabel('Time (hours)', fontsize=12)
ax2.set_ylabel('Mean Depth (m)', fontsize=12)
ax2.set_title('Mean Water Depth Over Time', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=10)

# Plot 3: Total volume over time
ax3 = axes[1, 0]
ax3.plot(time_hours, np.array(total_volume_data)/1e6, 'purple', linewidth=2)
ax3.axvline(x=config.rainfall_duration_hours, color='r', linestyle='--', 
            linewidth=1.5, label='Rain stops', alpha=0.7)
ax3.set_xlabel('Time (hours)', fontsize=12)
ax3.set_ylabel('Total Water Volume (million m³)', fontsize=12)
ax3.set_title('Total Water Volume Over Time', fontsize=13, fontweight='bold')
ax3.grid(True, alpha=0.3)
ax3.legend(fontsize=10)

# Plot 4: Timestep over time
ax4 = axes[1, 1]
ax4.plot(time_hours, timestep_data, 'orange', linewidth=2)
ax4.axvline(x=config.rainfall_duration_hours, color='r', linestyle='--', 
            linewidth=1.5, label='Rain stops', alpha=0.7)
ax4.set_xlabel('Time (hours)', fontsize=12)
ax4.set_ylabel('Timestep (s)', fontsize=12)
ax4.set_title('Adaptive Timestep Over Time', fontsize=13, fontweight='bold')
ax4.grid(True, alpha=0.3)
ax4.legend(fontsize=10)

plt.tight_layout()
timeseries_plot = os.path.join(config.output_dir, 'timeseries_plots.png')
plt.savefig(timeseries_plot, dpi=150, bbox_inches='tight')
print(f"✓ Time series plots saved: {timeseries_plot}")
plt.show()

# Print summary statistics
print("\n" + "="*70)
print("SUMMARY STATISTICS")
print("="*70)
total_rainfall_volume = config.rainfall_intensity_mm_hr * domain.get_area() / 1e6
print(f"Total rainfall input: {total_rainfall_volume:.2f} million m³")
print(f"Peak water depth: {max(max_depth_data):.4f} m = {max(max_depth_data)*1000:.2f} mm")
print(f"Peak volume: {max(total_volume_data)/1e6:.2f} million m³")
print(f"Final volume: {total_volume_data[-1]/1e6:.2f} million m³")
retention_pct = 100 * total_volume_data[-1] / (total_rainfall_volume * 1e6)
print(f"Water retention: {retention_pct:.1f}% (rest drained)")
print("="*70)

## Animation from Simulation Output

This cell creates an animation from the saved ANUGA output (.sww file).

In [ ]:
# RESUME SIMULATION FROM CHECKPOINT
import pickle
import glob

# Find latest checkpoint
checkpoint_files = glob.glob(os.path.join(config.output_dir, 'checkpoint_*.pkl'))
if not checkpoint_files:
    print("No checkpoint files found!")
else:
    # Get the latest checkpoint
    latest_checkpoint = max(checkpoint_files, key=os.path.getctime)
    print(f"Loading checkpoint: {latest_checkpoint}")
    
    with open(latest_checkpoint, 'rb') as f:
        checkpoint_data = pickle.load(f)
    
    # Restore state
    resume_time = checkpoint_data['time']
    domain.set_quantity('stage', checkpoint_data['stage'], location='centroids')
    domain.set_quantity('xmomentum', checkpoint_data['xmom'], location='centroids')
    domain.set_quantity('ymomentum', checkpoint_data['ymom'], location='centroids')
    domain.timestep = checkpoint_data['timestep']
    
    print(f"✓ Checkpoint loaded from t={resume_time/3600:.2f} hours")
    print(f"  You can now continue the simulation from this point")
    print(f"  Adjust finaltime in the simulation cell to continue further")

## Resume from Checkpoint (Optional)

If your simulation was interrupted, you can resume from the last checkpoint:

In [ ]:
# Visualize the triangulated mesh
import matplotlib.pyplot as plt
import matplotlib.tri as tri

# Get mesh coordinates and triangles
points = domain.get_vertex_coordinates()
triangles = domain.get_triangles()

# Reshape points to x, y coordinates
x = points[:, 0]
y = points[:, 1]

# Create triangulation for plotting
triang = tri.Triangulation(x, y, triangles)

# Create figure with subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Plot 1: Mesh triangulation
ax1.triplot(triang, 'b-', linewidth=0.3, alpha=0.5)
ax1.set_xlabel('Easting (m)', fontsize=12)
ax1.set_ylabel('Northing (m)', fontsize=12)
ax1.set_title(f'Triangulated Mesh\n({domain.get_number_of_triangles():,} triangles)', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.set_aspect('equal')

# Plot 2: Elevation on mesh
elevation_values = domain.get_quantity('elevation').get_values(location='centroids')
tcf = ax2.tripcolor(triang, elevation_values, cmap='terrain', shading='flat')
ax2.set_xlabel('Easting (m)', fontsize=12)
ax2.set_ylabel('Northing (m)', fontsize=12)
ax2.set_title('Elevation on Mesh', fontsize=14, fontweight='bold')
ax2.set_aspect('equal')
cbar = plt.colorbar(tcf, ax=ax2)
cbar.set_label('Elevation (m)', fontsize=11)

plt.tight_layout()
plt.show()

print(f"\nMesh Statistics:")
print(f"  Total triangles: {domain.get_number_of_triangles():,}")
print(f"  Total vertices: {len(x):,}")
print(f"  Domain area: {domain.get_area():,.2f} m²")
print(f"  Domain area: {domain.get_area()/1e6:.2f} km²")

In [ ]:
# VERIFY GPU SETUP, FORCE GPU INTERFACE, AND PREWARM KERNELS
# This cell ensures the GPU interface is created even if set_multiprocessor_mode
# fell back to CPU previously. It also prewarms GPU kernels so they are compiled
# and resident prior to the long simulation.

# Quick config knobs (adjust if you want heavier prewarm/load)
config.prewarm_iterations = getattr(config, 'prewarm_iterations', 8)   # number of small kernel calls to prewarm
config.prewarm_edge_calls = getattr(config, 'prewarm_edge_calls', 2)   # call extrapolation kernels this many times
config.force_gpu_init = getattr(config, 'force_gpu_init', True)       # force manual GPU init if set_multiprocessor_mode didn't

# Check GPU availability regardless of config.use_gpu (which might have been disabled by fallback)
if GPU_AVAILABLE:
    print('='*70)
    print('GPU VERIFICATION & FORCED INIT')
    print('='*70)
    import cupy as cp
    
    if not config.use_gpu:
        print("⚠ Note: config.use_gpu was False (likely due to fallback). Attempting to FORCE GPU...")
        config.use_gpu = True

    # Try to set multiprocessor mode first (preferred path)
    try:
        domain.set_multiprocessor_mode(2)
    except Exception as e:
        print(f'Warning: domain.set_multiprocessor_mode(2) raised: {e}')

    # If domain did not end up in GPU mode or no gpu_interface exists, try manual creation
    if getattr(domain, 'get_multiprocessor_mode', lambda: None)() != 2 or getattr(domain, 'gpu_interface', None) is None:
        if not config.force_gpu_init:
            print('Multiprocessor mode not GPU and force init disabled - skipping manual GPU init')
        else:
            print('Multiprocessor mode is not 2 or gpu_interface missing — attempting manual GPU interface creation...')
            try:
                from anuga.shallow_water.sw_domain_cuda import GPU_interface
                domain.gpu_interface = GPU_interface(domain)
                # allocate arrays and compile kernels
                print('Allocating GPU arrays (this may take a moment)...')
                domain.gpu_interface.allocate_gpu_arrays()
                print('Compiling GPU kernels (this may take a moment)...')
                try:
                    domain.gpu_interface.compile_gpu_kernels()
                except Exception as ce:
                    # Compilation may expect different path to cuda_anuga.cu on your system
                    print(f'Warning: kernel compilation failed: {ce}')
                    print('Kernels may still be available if precompiled cubin is found elsewhere.')
                # force set the mode marker
                domain.multiprocessor_mode = 2
                print('✓ Manual GPU interface created and arrays allocated (if no exceptions)')
            except Exception as e:
                import traceback
                print(f'✗ Manual GPU init failed: {e}')
                traceback.print_exc()
                print('Falling back to CPU mode')
                try:
                    import multiprocessing
                    num_threads = multiprocessing.cpu_count()
                    domain.set_omp_num_threads(num_threads)
                    print(f"✓ CPU mode: Using {num_threads} OpenMP threads")
                except:
                    pass
                config.use_gpu = False

    # If we have a gpu_interface, do a prewarm sequence to compile/load kernels and allocate memory
    if getattr(domain, 'gpu_interface', None) is not None and getattr(domain, 'get_multiprocessor_mode', lambda: 1)() == 2:
        gi = domain.gpu_interface

        # Quick memory snapshot before prewarm
        try:
            mempool = cp.get_default_memory_pool()
            print('\nGPU memory before prewarm:')
            print(f"  Used: {mempool.used_bytes()/1e6:.1f} MB | Total tracked: {mempool.total_bytes()/1e6:.1f} MB")
        except Exception:
            pass

        # Ensure GPU arrays reflect current CPU values
        try:
            gi.cpu_to_gpu_centroid_values()
            gi.cpu_to_gpu_edge_values()
            gi.cpu_to_gpu_boundary_values()
        except Exception as e:
            print(f'Warning: failed to copy CPU->GPU initial arrays: {e}')

        # Prewarm compute_fluxes kernel a number of times
        print('\nPrewarming flux kernel...')
        prewarm_iters = max(1, int(config.prewarm_iterations))
        for i in range(prewarm_iters):
            try:
                # Use transfer_gpu_results=False for prewarm to avoid expensive host transfers
                _ = gi.compute_fluxes_ext_central_kernel(domain.timestep, transfer_from_cpu=False, transfer_gpu_results=False)
            except Exception as e:
                # If transfer_from_cpu=False fails (some arrays not set), try with transfer_from_cpu=True
                try:
                    _ = gi.compute_fluxes_ext_central_kernel(domain.timestep, transfer_from_cpu=True, transfer_gpu_results=False)
                except Exception as e2:
                    print(f'Prewarm iteration {i} failed: {e2}')
                    break
        print('Done prewarming flux kernel')

        # Prewarm extrapolation kernels (these are used frequently)
        print('\nPrewarming extrapolation kernels...')
        for j in range(max(1, int(config.prewarm_edge_calls))):
            try:
                gi.extrapolate_second_order_edge_sw_kernel(transfer_from_cpu=False, transfer_gpu_results=False)
            except Exception:
                try:
                    gi.extrapolate_second_order_edge_sw_kernel(transfer_from_cpu=True, transfer_gpu_results=False)
                except Exception as e:
                    print(f'Extrapolation prewarm failed: {e}')
                    break
        print('Done prewarming extrapolation kernels')

        # Show memory after prewarm
        try:
            mempool_after = cp.get_default_memory_pool()
            print('\nGPU memory after prewarm:')
            print(f"  Used: {mempool_after.used_bytes()/1e6:.1f} MB | Total tracked: {mempool_after.total_bytes()/1e6:.1f} MB")
        except Exception:
            pass

        print('\nPrewarm complete — now proceed to full simulation.\n')
        print('💡 TIP: While running the full simulation, open a terminal and run `nvidia-smi -l 1` to watch GPU usage live.')

    else:
        print('\nGPU interface not present or not in GPU mode; cannot prewarm kernels')
        print(f'  Multiprocessor mode: {getattr(domain, "get_multiprocessor_mode", lambda: 1)()}')
        print(f'  gpu_interface: {getattr(domain, "gpu_interface", None)}')

else:
    print('GPU not enabled or not available - skipping verification/prewarm')
    print(f'Config use_gpu: {config.use_gpu}')
    print(f'GPU available: {GPU_AVAILABLE}')
    if hasattr(domain, 'get_multiprocessor_mode'):
        print(f'Multiprocessor mode: {domain.get_multiprocessor_mode()}')

## GPU Module Monkey-Patch

The installed anuga lacks `sw_domain_cuda`. Import it from the local repo.

In [ ]:
# MONKEY-PATCH GPU MODULE FROM LOCAL REPO
# The installed anuga package doesn't have sw_domain_cuda, so load it from local repo

print('--- GPU MODULE MONKEY-PATCH ---')
import sys, importlib.util, pathlib

# Check if anuga.shallow_water already has sw_domain_cuda
try:
    from anuga.shallow_water import sw_domain_cuda
    print('✓ anuga.shallow_water.sw_domain_cuda already available')
except ImportError:
    print('anuga.shallow_water.sw_domain_cuda not found in installed package')
    print('Loading from local repo...')
    
    local_repo = pathlib.Path(r'e:\4th year\ANUGA-ing\anuga_core-main').resolve()
    sw_domain_cuda_path = local_repo / 'anuga' / 'shallow_water' / 'sw_domain_cuda.py'
    
    if not sw_domain_cuda_path.exists():
        raise FileNotFoundError(f'sw_domain_cuda.py not found at: {sw_domain_cuda_path}')
    
    print(f'  Found: {sw_domain_cuda_path}')
    
    # Load the module
    spec = importlib.util.spec_from_file_location(
        'anuga.shallow_water.sw_domain_cuda',
        str(sw_domain_cuda_path)
    )
    sw_domain_cuda = importlib.util.module_from_spec(spec)
    
    # Register it in sys.modules so imports work
    sys.modules['anuga.shallow_water.sw_domain_cuda'] = sw_domain_cuda
    
    # Execute the module
    try:
        spec.loader.exec_module(sw_domain_cuda)
        print(f'  ✓ Loaded sw_domain_cuda from local repo')
        print(f'  ✓ GPU_interface class available: {hasattr(sw_domain_cuda, "GPU_interface")}')
    except Exception as e:
        print(f'  ✗ Failed to load sw_domain_cuda: {e}')
        import traceback
        traceback.print_exc()
        raise

print('✓ GPU module monkey-patch complete')
